# Lab: Storing and Retrieving Data with SQLite (sqlite3 and SQLAlchemy)

In the earlier lab we fetched data from the web and saved it as CSV files. Today we store data in a **database** and get it back using **SQL**.

| Part | Tool | You will learn |
|---|---|---|
| A | sqlite3 | Create a table, insert, select |
| B | SQLAlchemy + pandas | Write a DataFrame into a database and read it back |

Libraries

| Library | Used for |
|---|---|
| sqlite3 | Built into Python. Talks to an SQLite database file |
| pandas | DataFrames, to_sql(), read_sql() |
| sqlalchemy | One common way to connect Python to many kinds of databases |


In [19]:
# write all your imports here
import sqlite3
import pandas as pd


### Our data: a small sample of college data

We will use three small tables.

| Table | Columns |
|---|---|
| students | roll_no, name, branch, city, marks |
| branches | branch, full_name, hod |
| courses | course_id, course_name, credits, branch |


In [2]:
students_data = [
    (1,  "Aarav",  "CSE", "Mathura", 78),
    (2,  "Diya",   "IT",  "Agra",    85),
    (3,  "Kabir",  "ECE", "Delhi",   62),
    (4,  "Meera",  "CSE", "Agra",    91),
    (5,  "Rohan",  "ME",  "Mathura", 55),
    (6,  "Sana",   "IT",  "Delhi",   73),
    (7,  "Vihaan", "CSE", "Aligarh", 66),
    (8,  "Isha",   "ECE", "Mathura", 88),
    (9,  "Arjun",  "ME",  "Agra",    47),
    (10, "Tara",   "IT",  "Mathura", 95),
    (11, "Yash",   "CSE", "Delhi",   59),
    (12, "Nisha",  "ECE", "Aligarh", 70),
]

branches_data = [
    ("CSE", "Computer Science and Engineering", "Dr. Sharma"),
    ("IT",  "Information Technology",           "Dr. Verma"),
    ("ECE", "Electronics and Communication",    "Dr. Singh"),
    ("ME",  "Mechanical Engineering",           "Dr. Gupta"),
]

print(len(students_data), "students")
print(students_data[0])

12 students
(1, 'Aarav', 'CSE', 'Mathura', 78)


## Part A: SQLite with sqlite3

SQLite keeps a whole database in **one file**. No server, no password.

Workflow:

**connect() → cursor() → execute() → commit() → close()**

| Method | Meaning |
|---|---|
| sqlite3.connect("file.db") | Open the database file (creates it if missing) |
| conn.cursor() | The object that runs SQL commands |
| cur.execute(sql) | Run one SQL command |
| conn.commit() | Save the changes permanently |
| cur.fetchall() / fetchone() | Get rows back after a SELECT |
| conn.close() | Close the connection |

SELECT only reads. INSERT, UPDATE and DELETE change data, so they need conn.commit().

### Task 1: Connect to a database

Open (create) a database file called college.db and make a cursor.

In [3]:
# connect to "college.db" and store in conn
conn = sqlite3.connect("college.db")
# create a cursor and store in cur
cur = conn.cursor()

### Task 2: Create a table

A table needs column names and data types.

| SQLite type | Python equivalent |
|---|---|
| INTEGER | int |
| REAL | float |
| TEXT | str |

PRIMARY KEY means every row has a different value in that column. `CREATE TABLE IF NOT EXISTS` means: no error if the table is already there.


In [4]:
# write a CREATE TABLE IF NOT EXISTS command for table students
# columns: roll_no INTEGER PRIMARY KEY, name TEXT, branch TEXT, city TEXT, marks INTEGER
cur.execute("""
create table if not exists students (
roll_no integer primary key,
name text,
branch text,
city text,
marks integer
)
""")
# run it with cur.execute() and commit
conn.commit()
# list tables: SELECT name FROM sqlite_master WHERE type='table'
# print cur.fetchall()

In [5]:
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
cur.fetchall()

[('students',)]

### Task 3: Insert one row

Use `?` placeholders and pass the values as a tuple.

```python
cur.execute("INSERT INTO students VALUES (?, ?, ?, ?, ?)", (values...))
```

You should see: 1 row inserted

In [9]:
# insert one test row: (101, "Test Student", "CSE", "Mathura", 50)
cur.execute("INSERT INTO students VALUES (?, ?, ?, ?, ?)", (102, "Aman", "CSE", "Mathura", 50))
# commit
conn.commit()
# print how many rows were inserted (cur.rowcount)
print(cur.rowcount)

1


### Task 4: Insert many rows

Do not write 12 INSERT commands. We already have a **list of tuples**, so use `cur.executemany()`.

`INSERT OR REPLACE` means: if the roll_no already exists, replace that row instead of giving an error. This lets you run the cell again safely.

You should see: 13 (12 students plus our test row)

In [10]:
# use cur.executemany() with INSERT OR REPLACE INTO students VALUES (?, ?, ?, ?, ?) and students_data
cur.executemany("insert or replace into students values(?, ?, ?, ?, ?)",students_data)
# commit
conn.commit()
# run SELECT COUNT(*) FROM students and print the result using fetchone()
print(cur.execute("select count(*) from students").fetchone())

(14,)


### Task 5: Read data with SELECT

| SQL | Meaning |
|---|---|
| SELECT * FROM t | All columns, all rows |
| WHERE | Filter rows |
| ORDER BY col DESC | Sort (ASC is default) |
| LIMIT n | Only first n rows |
| LIKE 'A%' | Text that starts with A |

fetchall() returns a **list of tuples**. fetchone() returns one tuple.

In [18]:
# select all columns of the first 5 rows and print each row using a loop
cur.execute("select * from students ")
for row in cur.fetchall():
    print(row)

print()
# select name and marks of students with marks >= 75, highest marks first, and print them
cur.execute("""
select name, marks from students where marks >= 75 order by marks desc
""")
for row in cur.fetchall():
    print(row)
# select the topper (hint: ORDER BY marks DESC LIMIT 1) and use fetchone()
print()
cur.execute("""
select name, marks from students order by marks desc limit 1
""")
print(cur.fetchone())
# select name and city of students whose name starts with the letter A (LIKE 'A%')
print()
cur.execute("""
select name, city from students where name like 'A%'
""")
for row in cur.fetchall():
    print(row)

(1, 'Aarav', 'CSE', 'Mathura', 78)
(2, 'Diya', 'IT', 'Agra', 85)
(3, 'Kabir', 'ECE', 'Delhi', 62)
(4, 'Meera', 'CSE', 'Agra', 91)
(5, 'Rohan', 'ME', 'Mathura', 55)
(6, 'Sana', 'IT', 'Delhi', 73)
(7, 'Vihaan', 'CSE', 'Aligarh', 66)
(8, 'Isha', 'ECE', 'Mathura', 88)
(9, 'Arjun', 'ME', 'Agra', 47)
(10, 'Tara', 'IT', 'Mathura', 95)
(11, 'Yash', 'CSE', 'Delhi', 59)
(12, 'Nisha', 'ECE', 'Aligarh', 70)
(101, 'Aman', 'CSE', 'Mathura', 50)
(102, 'Aman', 'CSE', 'Mathura', 50)

('Tara', 95)
('Meera', 91)
('Isha', 88)
('Diya', 85)
('Aarav', 78)

('Tara', 95)

('Aarav', 'Mathura')
('Arjun', 'Agra')
('Aman', 'Mathura')
('Aman', 'Mathura')


### Task **6**: Get the result as a DataFrame

Tuples are hard to work with. `pd.read_sql_query(sql, conn)` runs the SQL and gives a **DataFrame** with proper column names.

You should see: a DataFrame with columns name, branch, marks.

In [21]:
# use pd.read_sql_query to get name, branch, marks of students from Mathura, highest marks first
data = pd.read_sql_query("""
select * from students
""", conn)
# store in mathura_df and display it
display(data)

,roll_no,name,branch,city,marks
0,1,Aarav,CSE,Mathura,78
1,2,Diya,IT,Agra,85
2,3,Kabir,ECE,Delhi,62
3,4,Meera,CSE,Agra,91
4,5,Rohan,ME,Mathura,55
5,6,Sana,IT,Delhi,73
6,7,Vihaan,CSE,Aligarh,66
7,8,Isha,ECE,Mathura,88
8,9,Arjun,ME,Agra,47
9,10,Tara,IT,Mathura,95


### Task 7: Close the connection

Always close the connection when you finish. After closing, cur and conn cannot be used again.

You should see: Closed

In [ ]:
# close the connection
# print Closed

## Part B: SQLAlchemy with pandas

SQLAlchemy connects Python to many databases (SQLite, MySQL, PostgreSQL) in the same way. Only the connection string changes.

Workflow:

**create_engine() → df.to_sql() → pd.read_sql()**

| Part | Meaning |
|---|---|
| create_engine("sqlite:///college.db") | The connection to the database file. Three slashes for a local file |
| df.to_sql("table", engine) | Write a whole DataFrame as a table |
| pd.read_sql(sql, engine) | Run SQL and get a DataFrame |
| text("SQL ... :name") | A safe SQL command with named placeholders |
| inspect(engine) | Look at the tables and columns in the database |

### Task 1: Create the engine

You should see: Engine(sqlite:///college.db)

In [ ]:
# create an engine for "sqlite:///college.db" and store in engine
# print engine

### Task 2: Write a DataFrame into the database

Here is a DataFrame of courses. Write it as a table.

`if_exists` decides what happens when the table already exists:

| Value | Meaning |
|---|---|
| "fail" | Give an error (default) |
| "replace" | Delete the old table and create a new one |
| "append" | Add the new rows to the old table |

Use index=False so the DataFrame index does not become a column.

You should see: 8

In [ ]:
courses_df = pd.DataFrame({
    "course_id":   [1, 2, 3, 4, 5, 6, 7, 8],
    "course_name": ["Python", "Data Structures", "DBMS", "Data Science",
                    "Digital Electronics", "Signals", "Thermodynamics", "Machine Design"],
    "credits":     [4, 4, 3, 4, 3, 3, 4, 3],
    "branch":      ["CSE", "CSE", "IT", "IT", "ECE", "ECE", "ME", "ME"],
})
courses_df

,course_id,course_name,credits,branch
0,1,Python,4,CSE
1,2,Data Structures,4,CSE
2,3,DBMS,3,IT
3,4,Data Science,4,IT
4,5,Digital Electronics,3,ECE
5,6,Signals,3,ECE
6,7,Thermodynamics,4,ME
7,8,Machine Design,3,ME


In [ ]:
# write courses_df to a table named courses, if_exists="replace", index=False
# to_sql returns the number of rows written, print it

### Task 3: Look inside the database

You should see: the table names (students, branches, courses) and the column names of courses.

In [ ]:
# create an inspector: insp = inspect(engine)
# print the list of table names using insp.get_table_names()
# print the column names and types of the courses table using insp.get_columns("courses")

### Task 4: Read with SQL into a DataFrame

You should see: all students, topper first, as a DataFrame.

In [ ]:
# read the whole students table sorted by marks (highest first) using pd.read_sql
# store in students_df and display it
# also display students_df.describe()

### Task 5: SQL with a parameter

`text()` lets you write a **named placeholder** with a colon, like `:branch`, and pass its value as a dictionary.

You should see: the courses of the IT branch.

In [ ]:
#read the course_name credits from courses table with branch IT passed as parameter

### Task 6: Create a new table from a query

Reading needs only engine.connect(). When you **change** data, use `engine.begin()`. It commits automatically when the block ends.

You should see: the number of students with marks of 80 or more (4).

In [ ]:
# use: with engine.begin() as conn:
# run: DROP TABLE IF EXISTS toppers
# run: CREATE TABLE toppers AS SELECT * FROM students WHERE marks >= 80
# then read SELECT COUNT(*) AS total FROM toppers using pd.read_sql and print it

### Task 7: Add new rows with append

You should see: 13 students after appending one.

In [ ]:
# make a DataFrame with one new student: roll_no 13, name "Pooja", branch "IT", city "Agra", marks 81
#   (use the same column names as the students table)
# append it to the students table using if_exists="append", index=False
# count the rows in students using pd.read_sql

Note: running the cell above a second time gives an error, because roll_no 13 already exists and it is a PRIMARY KEY. That is the database protecting your data.

### Task 8: Check with sqlite3 again

Both tools use the **same file**. Open college.db with sqlite3 and list all tables.

You should see: students, branches, courses, toppers.

In [ ]:
# connect to college.db using sqlite3
# list all table names from sqlite_master
# close the connection